# Phase 2: Full FT CaseHOLD + Random Label Baseline

AutoDL H800 80GB — 补充实验

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E9  | Full FT CaseHOLD Qwen  | Full FT | CaseHOLD | Qwen2.5-1.5B |
| E10 | Full FT CaseHOLD Llama | Full FT | CaseHOLD | Llama-3.2-1B |
| E11 | Random BillSum Qwen    | LoRA (shuffled labels) | BillSum  | Qwen2.5-1.5B |
| E12 | Random BillSum Llama   | LoRA (shuffled labels) | BillSum  | Llama-3.2-1B |
| E13 | Random CaseHOLD Qwen   | LoRA (shuffled labels) | CaseHOLD | Qwen2.5-1.5B |
| E14 | Random CaseHOLD Llama  | LoRA (shuffled labels) | CaseHOLD | Llama-3.2-1B |

## 0. 环境准备

In [1]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)

# 软链到数据盘
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

  ✓ symlink exists: /root/MLP/outputs -> /root/autodl-tmp/outputs
  ⚠ /root/MLP/logs is a real dir, skipping
  ⚠ /root/MLP/data is a real dir, skipping
/root/MLP
README.md	  autodl_run_phase2.ipynb  data  models   results  src
autodl_run.ipynb  configs		   logs  outputs  scripts  wandb
Python: /root/miniconda3/bin/python
HF_HOME: /root/autodl-tmp/hf_cache
HF_ENDPOINT: https://hf-mirror.com
WANDB_MODE: offline


In [2]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub
print('\n✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel')

Looking in indexes: http://mirrors.aliyun.com/pypi/simple

✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel


In [3]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA H800 PCIe


In [4]:
# HuggingFace 登录 (Llama 需要)
from huggingface_hub import login
login()
print('HuggingFace 登录成功')

HuggingFace 登录成功


## 1. 数据验证

In [5]:
import os
files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]
for f in files:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else -1
    status = f'✓ {size} KB' if size >= 0 else '✗ 缺失！'
    print(f'{status}  {f}')

✓ 282658 KB  data/billsum/train_sft.jsonl
✓ 15130 KB  data/billsum/val_sft.jsonl
✓ 51177 KB  data/billsum/test_us_sft.jsonl
✓ 12753 KB  data/billsum/test_ca_sft.jsonl
✓ 115681 KB  data/casehold/train_mc.jsonl
✓ 19727 KB  data/casehold/validation_mc.jsonl
✓ 19716 KB  data/casehold/test_mc.jsonl


---
## 2. Full Fine-Tuning — CaseHOLD

使用 `train.py`（无 `lora` 节时自动全量微调）。

### E9: Full FT — CaseHOLD × Qwen2.5-1.5B

In [6]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/full_casehold_qwen.yaml 2>&1 | tee logs/full_casehold_qwen.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_casehold_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/casehold/train_mc.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 479.11it/s]
Full fine-tuning: 1.54B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 5223/5223 [00:05<0

In [7]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_casehold_qwen.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/full_casehold_qwen/predictions_test.jsonl --output results/casehold/full_qwen_test.json
print('评估完成')
!cat results/casehold/full_qwen_test.json

Config     : configs/full_casehold_qwen.yaml
Task       : classification
Split      : test  →  /root/MLP/data/casehold/test_mc.jsonl
Output     : /root/MLP/outputs/full_casehold_qwen
Batch size : 16
Loading full fine-tuned model: /root/MLP/outputs/full_casehold_qwen/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 435.20it/s]
Loaded 5,221 test records
The following generation flags are not valid and may be ignored: ['temperature', 

### E10: Full FT — CaseHOLD × Llama-3.2-1B

In [8]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_casehold_llama.yaml 2>&1 | tee logs/full_casehold_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_casehold_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/casehold/train_mc.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 258.30it/s]
Full fine-tuning: 1.24B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 5223/5223 [

In [9]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_casehold_llama.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/full_casehold_llama/predictions_test.jsonl --output results/casehold/full_llama_test.json
print('评估完成')
!cat results/casehold/full_llama_test.json

Config     : configs/full_casehold_llama.yaml
Task       : classification
Split      : test  →  /root/MLP/data/casehold/test_mc.jsonl
Output     : /root/MLP/outputs/full_casehold_llama
Batch size : 16
Loading full fine-tuned model: /root/MLP/outputs/full_casehold_llama/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 240.72it/s]
Loaded 5,221 test records
The following generation flags are not valid and may be ignored: ['temperature

---
## 3. Random Label Baseline (LoRA)

将训练集的 output 打乱（input 不变），使 input→output 映射完全随机。  
如果模型确实学到了任务知识，random label 训练后在真实测试集上的表现应大幅低于正常训练。

### 3.0 生成 Random Label 数据

In [10]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label Datasets ===')
shuffle_labels('data/billsum/train_sft.jsonl',       'data/billsum/train_sft_random.jsonl')
shuffle_labels('data/billsum/val_sft.jsonl',         'data/billsum/val_sft_random.jsonl')
shuffle_labels('data/casehold/train_mc.jsonl',       'data/casehold/train_mc_random.jsonl')
shuffle_labels('data/casehold/validation_mc.jsonl',  'data/casehold/validation_mc_random.jsonl')
print('\nDone. Test files are NOT shuffled (evaluate on real data).')

=== Generating Random Label Datasets ===
  ✓ data/billsum/train_sft_random.jsonl: 15,988 records (labels shuffled)
  ✓ data/billsum/val_sft_random.jsonl: 842 records (labels shuffled)
  ✓ data/casehold/train_mc_random.jsonl: 30,652 records (labels shuffled)
  ✓ data/casehold/validation_mc_random.jsonl: 5,223 records (labels shuffled)

Done. Test files are NOT shuffled (evaluate on real data).


In [11]:
# 验证 mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (BillSum ~1.0, CaseHOLD ~0.8):')
print(f'  BillSum train:  {mismatch_rate("data/billsum/train_sft.jsonl", "data/billsum/train_sft_random.jsonl"):.4f}')
print(f'  BillSum val:    {mismatch_rate("data/billsum/val_sft.jsonl", "data/billsum/val_sft_random.jsonl"):.4f}')
print(f'  CaseHOLD train: {mismatch_rate("data/casehold/train_mc.jsonl", "data/casehold/train_mc_random.jsonl"):.4f}')
print(f'  CaseHOLD val:   {mismatch_rate("data/casehold/validation_mc.jsonl", "data/casehold/validation_mc_random.jsonl"):.4f}')

Mismatch rates (BillSum ~1.0, CaseHOLD ~0.8):
  BillSum train:  1.0000
  BillSum val:    0.9988
  CaseHOLD train: 0.8006
  CaseHOLD val:   0.8072


### E11: Random Label LoRA — BillSum × Qwen2.5-1.5B

In [12]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_billsum_qwen.yaml 2>&1 | tee logs/random_billsum_qwen.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_billsum_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/billsum/train_sft_random.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft_random.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 472.43it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Token

In [13]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_qwen.yaml --split test_us --batch_size 8
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_qwen.yaml --split test_ca --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_qwen/predictions_test_us.jsonl --output results/billsum/random_qwen_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_qwen/predictions_test_ca.jsonl --output results/billsum/random_qwen_test_ca.json
print('评估完成')
!cat results/billsum/random_qwen_test_us.json

Config     : configs/random_billsum_qwen.yaml
Task       : summarization
Split      : test_us  →  /root/MLP/data/billsum/test_us_sft.jsonl
Output     : /root/MLP/outputs/random_billsum_qwen
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 227.35it/s]
Loading LoRA adapter: /root/MLP/outputs/random_billsum_qwen/final_adapter
Loaded 2,905 test records
The following generation flags are

### E12: Random Label LoRA — BillSum × Llama-3.2-1B

In [14]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_billsum_llama.yaml 2>&1 | tee logs/random_billsum_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_billsum_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/billsum/train_sft_random.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft_random.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:01<00:00, 140.31it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets..

In [15]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_llama.yaml --split test_us --batch_size 8
!{sys.executable} -u src/evaluate/inference.py --config configs/random_billsum_llama.yaml --split test_ca --batch_size 8
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_llama/predictions_test_us.jsonl --output results/billsum/random_llama_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/random_billsum_llama/predictions_test_ca.jsonl --output results/billsum/random_llama_test_ca.json
print('评估完成')
!cat results/billsum/random_llama_test_us.json

Config     : configs/random_billsum_llama.yaml
Task       : summarization
Split      : test_us  →  /root/MLP/data/billsum/test_us_sft.jsonl
Output     : /root/MLP/outputs/random_billsum_llama
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 306.19it/s]
Loading LoRA adapter: /root/MLP/outputs/random_billsum_llama/final_adapter
Loaded 2,905 test records
The following generation 

### E13: Random Label LoRA — CaseHOLD × Qwen2.5-1.5B

In [16]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_casehold_qwen.yaml 2>&1 | tee logs/random_casehold_qwen.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_casehold_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/casehold/train_mc_random.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc_random.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 376.16it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets.

In [17]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_casehold_qwen.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/random_casehold_qwen/predictions_test.jsonl --output results/casehold/random_qwen_test.json
print('评估完成')
!cat results/casehold/random_qwen_test.json

Config     : configs/random_casehold_qwen.yaml
Task       : classification
Split      : test  →  /root/MLP/data/casehold/test_mc.jsonl
Output     : /root/MLP/outputs/random_casehold_qwen
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 425.06it/s]
Loading LoRA adapter: /root/MLP/outputs/random_casehold_qwen/final_adapter
Loaded 5,221 test records
The following generation flags are 

### E14: Random Label LoRA — CaseHOLD × Llama-3.2-1B

In [18]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_casehold_llama.yaml 2>&1 | tee logs/random_casehold_llama.log
print('训练完成')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_casehold_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/casehold/train_mc_random.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc_random.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 221.47it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing da

In [19]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_casehold_llama.yaml --split test --batch_size 16
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/random_casehold_llama/predictions_test.jsonl --output results/casehold/random_llama_test.json
print('评估完成')
!cat results/casehold/random_llama_test.json

Config     : configs/random_casehold_llama.yaml
Task       : classification
Split      : test  →  /root/MLP/data/casehold/test_mc.jsonl
Output     : /root/MLP/outputs/random_casehold_llama
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 222.02it/s]
Loading LoRA adapter: /root/MLP/outputs/random_casehold_llama/final_adapter
Loaded 5,221 test records
The following generation f

---
## 4. 汇总所有结果

In [20]:
import json, os

results = [
    ('full_casehold_qwen',    'results/casehold/full_qwen_test.json'),
    ('full_casehold_llama',   'results/casehold/full_llama_test.json'),
    ('random_billsum_qwen  (US)', 'results/billsum/random_qwen_test_us.json'),
    ('random_billsum_qwen  (CA)', 'results/billsum/random_qwen_test_ca.json'),
    ('random_billsum_llama (US)', 'results/billsum/random_llama_test_us.json'),
    ('random_billsum_llama (CA)', 'results/billsum/random_llama_test_ca.json'),
    ('random_casehold_qwen',  'results/casehold/random_qwen_test.json'),
    ('random_casehold_llama', 'results/casehold/random_llama_test.json'),
]

print(f'{"实验":<30} {"指标":<15} {"值":>8}')
print('-' * 55)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<30} {"未完成":<15}')
        continue
    with open(path) as f:
        d = json.load(f)
    if 'rouge2' in d:
        val = d['rouge2']['mean'] if isinstance(d['rouge2'], dict) else d['rouge2']
        print(f'{name:<30} {"rouge2":<15} {val:>8.4f}')
    elif 'accuracy' in d:
        val = d['accuracy'] if isinstance(d['accuracy'], (int, float)) else d['accuracy']['mean']
        print(f'{name:<30} {"accuracy":<15} {val:>8.4f}')

实验                             指标                     值
-------------------------------------------------------
full_casehold_qwen             accuracy          0.8339
full_casehold_llama            accuracy          0.8632
random_billsum_qwen  (US)      rouge2            0.0161
random_billsum_qwen  (CA)      rouge2            0.0022
random_billsum_llama (US)      rouge2            0.0292
random_billsum_llama (CA)      rouge2            0.0144
random_casehold_qwen           accuracy          0.2300
random_casehold_llama          accuracy          0.2099
